<a href="https://colab.research.google.com/github/soumaya-lamrhari/customer-conversion-prediction/blob/main/notebooks/3_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Conversion Modeling

## Objective

In this notebook, I train and compare machine learning models to predict customer conversion.

I first use Logistic Regression as a baseline model, then train an XGBoost model to capture more complex relationships in the data. The models will be evaluated using metrics appropriate for the imbalanced target.

In [1]:
!pip install -q ucimlrepo

import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

# Load the original Bank Marketing dataset
bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features.copy()
y = bank_marketing.data.targets.copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (45211, 16)
Target shape: (45211, 1)


## Data Preparation

In [2]:
# Remove duration because it would cause data leakage
X = X.drop(columns=["duration"])

# Keep meaningful missing values as separate categories
X["poutcome"] = X["poutcome"].fillna("no_previous_contact")
X["contact"] = X["contact"].fillna("unknown")
X["job"] = X["job"].fillna("unknown")
X["education"] = X["education"].fillna("unknown")

# Rename the column to reflect what the values actually represent
X = X.rename(columns={"day_of_week": "day_of_month"})

# Convert the target to binary values
y = y["y"].map({"yes": 1, "no": 0})

print("Features shape:", X.shape)
print("Missing values:", X.isna().sum().sum())
print("Conversion rate:", round(y.mean() * 100, 2), "%")

Features shape: (45211, 15)
Missing values: 0
Conversion rate: 11.7 %


In [3]:
from sklearn.model_selection import train_test_split

# Keep the same conversion distribution in the training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("Training conversion rate:", round(y_train.mean() * 100, 2), "%")
print("Test conversion rate:", round(y_test.mean() * 100, 2), "%")

Training set: (36168, 15)
Test set: (9043, 15)
Training conversion rate: 11.7 %
Test conversion rate: 11.7 %


In [5]:
# Separate numerical and categorical features for preprocessing
numerical_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

Numerical features: ['age', 'balance', 'day_of_month', 'campaign', 'pdays', 'previous']
Categorical features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


## Logistic Regression Baseline

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Encode categorical features and scale numerical features for Logistic Regression
logistic_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numerical_features)
    ]
)

logistic_preprocessor

ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['job', 'marital', 'education', 'default',
                                  'housing', 'loan', 'contact', 'month',
                                  'poutcome']),
                                ('num', StandardScaler(),
                                 ['age', 'balance', 'day_of_month', 'campaign',
                                  'pdays', 'previous'])])

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Combine preprocessing and Logistic Regression into one workflow
logistic_model = Pipeline(
    steps=[
        ("preprocessor", logistic_preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

logistic_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'poutcome']),
                                                 ('num', StandardScaler(),
                                                  ['age', 'balance',
                                                   'day_of_month', 'campaign',
                                                   'pdays', 'previous'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [8]:
logistic_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'poutcome']),
                                                 ('num', StandardScaler(),
                                                  ['age', 'balance',
                                                   'day_of_month', 'campaign',
                                                   'pdays', 'previous'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [9]:
# Predict conversion probabilities for the test customers
y_prob_logistic = logistic_model.predict_proba(X_test)[:, 1]

print(y_prob_logistic[:10])

[0.01323245 0.02565882 0.03579249 0.01729891 0.13251239 0.04812879
 0.07051483 0.84028351 0.1078987  0.09633624]


In [10]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Convert probabilities to class predictions using the default 0.50 threshold
y_pred_logistic = (y_prob_logistic >= 0.50).astype(int)

print("Accuracy:", round(accuracy_score(y_test, y_pred_logistic), 3))
print("Precision:", round(precision_score(y_test, y_pred_logistic), 3))
print("Recall:", round(recall_score(y_test, y_pred_logistic), 3))
print("F1 Score:", round(f1_score(y_test, y_pred_logistic), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_logistic), 3))
print("PR-AUC:", round(average_precision_score(y_test, y_prob_logistic), 3))

Accuracy: 0.893
Precision: 0.663
Recall: 0.179
F1 Score: 0.281
ROC-AUC: 0.772
PR-AUC: 0.411


### Baseline Results

Logistic Regression was used as a baseline to provide a simple model for comparison with XGBoost.

At the default 0.50 threshold, the model achieved high precision (66.3%) but low recall (17.9%), meaning that its positive predictions were often correct, but it identified only a small proportion of the customers who actually converted.

The PR-AUC of 0.411 is well above the 0.117 no-skill baseline, which corresponds to the 11.7% conversion rate in the dataset. This suggests that the model has useful ability to identify and rank potential converters. The classification threshold will be examined later after comparing the baseline with XGBoost.

## XGBoost Model

In [11]:
# Encode categorical features and keep numerical features unchanged for XGBoost
xgb_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

xgb_preprocessor

ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['job', 'marital', 'education', 'default',
                                  'housing', 'loan', 'contact', 'month',
                                  'poutcome']),
                                ('num', 'passthrough',
                                 ['age', 'balance', 'day_of_month', 'campaign',
                                  'pdays', 'previous'])])

In [12]:
!pip install -q xgboost

from xgboost import XGBClassifier

# Create the XGBoost classifier
xgb_classifier = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42,
    eval_metric="logloss"
)

In [13]:
# Combine preprocessing and XGBoost into one workflow
xgb_model = Pipeline(
    steps=[
        ("preprocessor", xgb_preprocessor),
        ("classifier", xgb_classifier)
    ]
)

xgb_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'poutcome']),
                                                 ('num', 'passthrough',
                                                  ['age', 'balance',
                                                   'day_of_month', 'campaign',
                                                   'pdays', 'previous'])])),
                ('classifier',
                 XGBClassifier(base_score=None, booster=None, callback...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=4, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=200, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [16]:
# Train the XGBoost model
xgb_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'poutcome']),
                                                 ('num', 'passthrough',
                                                  ['age', 'balance',
                                                   'day_of_month', 'campaign',
                                                   'pdays', 'previous'])])),
                ('classifier',
                 XGBClassifier(base_score=None, booster=None, callback...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=4, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=200, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [17]:
# Predict conversion probabilities for the test customers
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print(y_prob_xgb[:10])

[0.02148509 0.03349024 0.03774751 0.03200116 0.11109037 0.0432662
 0.05250799 0.7854805  0.09201479 0.07450187]


In [18]:
# Convert probabilities to class predictions using the default 0.50 threshold
y_pred_xgb = (y_prob_xgb >= 0.50).astype(int)

print("Accuracy:", round(accuracy_score(y_test, y_pred_xgb), 3))
print("Precision:", round(precision_score(y_test, y_pred_xgb), 3))
print("Recall:", round(recall_score(y_test, y_pred_xgb), 3))
print("F1 Score:", round(f1_score(y_test, y_pred_xgb), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_xgb), 3))
print("PR-AUC:", round(average_precision_score(y_test, y_prob_xgb), 3))

Accuracy: 0.895
Precision: 0.665
Recall: 0.216
F1 Score: 0.325
ROC-AUC: 0.804
PR-AUC: 0.465


### XGBoost Results

XGBoost improved performance compared with the Logistic Regression baseline. At the default 0.50 threshold, recall increased from 17.9% to 21.6%, while precision remained similar at 66.5%.

The PR-AUC increased from 0.411 to 0.465 and the ROC-AUC increased from 0.772 to 0.804, indicating that XGBoost provides better overall ability to identify and rank potential converters.

## Model Comparison

In [19]:
# Compare the baseline and XGBoost performance
model_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC", "PR-AUC"],
    "Logistic Regression": [0.893, 0.663, 0.179, 0.281, 0.772, 0.411],
    "XGBoost": [0.895, 0.665, 0.216, 0.325, 0.804, 0.465]
})

model_comparison

,Metric,Logistic Regression,XGBoost
0,Accuracy,0.893,0.895
1,Precision,0.663,0.665
2,Recall,0.179,0.216
3,F1 Score,0.281,0.325
4,ROC-AUC,0.772,0.804
5,PR-AUC,0.411,0.465


In [20]:
# Create a validation set for threshold selection
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

print("Training subset:", X_train_sub.shape)
print("Validation set:", X_val.shape)

print("Training conversion rate:", round(y_train_sub.mean() * 100, 2), "%")
print("Validation conversion rate:", round(y_val.mean() * 100, 2), "%")

Training subset: (28934, 15)
Validation set: (7234, 15)
Training conversion rate: 11.7 %
Validation conversion rate: 11.69 %


In [21]:
# Train XGBoost using only the training subset
xgb_model.fit(X_train_sub, y_train_sub)

# Predict conversion probabilities on the validation set
y_prob_val = xgb_model.predict_proba(X_val)[:, 1]

print(y_prob_val[:10])

[0.12177053 0.10585497 0.5483173  0.02851559 0.03234314 0.27323356
 0.3373515  0.13079208 0.1830272  0.02188582]


In [22]:
# Compare precision and recall across different classification thresholds
thresholds = [0.10, 0.20, 0.30, 0.40, 0.50]

threshold_results = []

for threshold in thresholds:
    y_pred_val = (y_prob_val >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_val, y_pred_val),
        "Recall": recall_score(y_val, y_pred_val),
        "F1 Score": f1_score(y_val, y_pred_val)
    })

threshold_results = pd.DataFrame(threshold_results).round(3)
threshold_results

,Threshold,Precision,Recall,F1 Score
0,0.1,0.281,0.708,0.403
1,0.2,0.473,0.545,0.506
2,0.3,0.538,0.422,0.473
3,0.4,0.611,0.313,0.414
4,0.5,0.663,0.221,0.332


In [23]:
# Evaluate the business impact of each threshold
business_results = []

total_customers = len(y_val)
total_converters = y_val.sum()

for threshold in thresholds:
    y_pred_val = (y_prob_val >= threshold).astype(int)

    customers_targeted = y_pred_val.sum()
    converters_captured = ((y_pred_val == 1) & (y_val == 1)).sum()

    business_results.append({
        "Threshold": threshold,
        "Customers Targeted": customers_targeted,
        "Targeted (%)": customers_targeted / total_customers * 100,
        "Converters Captured": converters_captured,
        "Capture Rate (%)": converters_captured / total_converters * 100
    })

business_results = pd.DataFrame(business_results).round(1)
business_results

,Threshold,Customers Targeted,Targeted (%),Converters Captured,Capture Rate (%)
0,0.1,2129,29.4,599,70.8
1,0.2,975,13.5,461,54.5
2,0.3,663,9.2,357,42.2
3,0.4,434,6.0,265,31.3
4,0.5,282,3.9,187,22.1


### Threshold Selection

I compared different thresholds to find a good balance between the number of customers targeted and the number of converters captured.

A threshold of 0.20 was selected because it targets 13.5% of customers while capturing 54.5% of actual converters.

This threshold will be used for the final model evaluation.

## Final Model Evaluation

The XGBoost model is retrained using the full training set and evaluated on the test set using the selected threshold of 0.20.

The final evaluation will measure both model performance and how effectively the model can prioritize potential converters.

In [24]:
# Retrain the final XGBoost model using the full training set
xgb_model.fit(X_train, y_train)

# Generate probabilities on the test set
y_prob_test = xgb_model.predict_proba(X_test)[:, 1]

# Apply the selected threshold
selected_threshold = 0.20
y_pred_test = (y_prob_test >= selected_threshold).astype(int)

In [25]:
# Evaluate the final model on the test set
print("Accuracy:", round(accuracy_score(y_test, y_pred_test), 3))
print("Precision:", round(precision_score(y_test, y_pred_test), 3))
print("Recall:", round(recall_score(y_test, y_pred_test), 3))
print("F1 Score:", round(f1_score(y_test, y_pred_test), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_test), 3))
print("PR-AUC:", round(average_precision_score(y_test, y_prob_test), 3))

Accuracy: 0.878
Precision: 0.479
Recall: 0.514
F1 Score: 0.496
ROC-AUC: 0.804
PR-AUC: 0.465


In [26]:
# Measure how many customers are targeted and converters are captured
customers_targeted = y_pred_test.sum()
total_customers = len(y_test)

converters_captured = ((y_pred_test == 1) & (y_test == 1)).sum()
total_converters = y_test.sum()

print("Customers targeted:", customers_targeted)
print("Targeted (%):", round(customers_targeted / total_customers * 100, 1))
print("Converters captured:", converters_captured)
print("Capture rate (%):", round(converters_captured / total_converters * 100, 1))

Customers targeted: 1135
Targeted (%): 12.6
Converters captured: 544
Capture rate (%): 51.4


## Modeling Summary

Using the selected threshold of 0.20, the final XGBoost model achieved a PR-AUC of 0.465 and identified 51.4% of actual converters.

The model targeted 12.6% of customers, showing that it can help focus marketing efforts on a smaller group of customers with higher conversion potential.